In [2]:
import olca_ipc as olca
import olca_schema
import pandas as pd
import uuid
import traceback
import os
import json # Added for JSON stringification

# Configuration
OLCA_PORT = 8080
LCIA_METHOD_NAME = "IPCC 2021 (ISO 14067)"
TARGET_IMPACT_CATEGORY = "IPCC 2021 (ISO 14067) - Total" # Still used for the specific CF column
PROCESS_NAMES_FILEPATH = "all_process_names.txt" # Path to your text file

def extract_result_safely(client, calc_result_obj): # target_category_name_str removed as we fetch all
    all_impact_values = {} # To store all impact categories and their values
    status = "Success"
    data_source_description = "Unknown source"

    if not calc_result_obj:
        print("                      ❌ Calculation result object is None or empty.")
        return {}, "Calculation Result Invalid/Empty" # Return empty dict

    if hasattr(calc_result_obj, 'error') and calc_result_obj.error:
        error_msg = getattr(calc_result_obj.error, 'error', str(calc_result_obj.error))
        print(f"                      ❌ Calculation failed with server error: {error_msg[:150]}")
        return {}, f"Calc Server Error: {error_msg[:100]}" # Return empty dict

    if not hasattr(calc_result_obj, 'uid') or not calc_result_obj.uid:
        if status == "Success": # Should ideally not happen if error above didn't catch it
            print("                      ❌ Calculation result object has no UID, might be error indicator.")
            return {}, "Calculation Result Invalid UID" # Return empty dict
        return {}, status # Return empty dict

    if hasattr(calc_result_obj, 'wait_until_ready'):
        try:
            print("                      ⏳ Waiting for calculation result to be ready...")
            calc_result_obj.wait_until_ready()
            if hasattr(calc_result_obj, 'error') and calc_result_obj.error:
                error_msg_after_wait = getattr(calc_result_obj.error, 'error', str(calc_result_obj.error))
                print(f"                      ⚠️ Calculation result (after wait) has error: {error_msg_after_wait[:150]}")
                return {}, f"Calc Error After Wait: {error_msg_after_wait[:100]}" # Return empty dict
        except Exception as e_wait:
            print(f"                      ⚠️ Error or issue during wait_until_ready: {e_wait}")
            print("                      📊 Attempting to read results anyway...")

    fetched_impact_data = None
    if hasattr(calc_result_obj, 'get_total_impacts'):
        try:
            print("                      🔍 Attempting to use 'get_total_impacts()'.")
            fetched_impact_data = calc_result_obj.get_total_impacts()
            data_source_description = "'get_total_impacts()'"
            if fetched_impact_data is not None:
                print(f"                      📊 '{data_source_description}' call successful, returned {'data' if len(fetched_impact_data) > 0 else 'an empty list'}.")
            else:
                print(f"                      📊 '{data_source_description}' returned None.")
        except Exception as e:
            print(f"                      ⚠️ Error calling 'get_total_impacts()': {e}")
            fetched_impact_data = None
            status = f"Error calling {data_source_description}"
    else:
        print("                      ℹ️  'get_total_impacts()' method not available.")
        if status == "Success": status = "get_total_impacts() not available"

    if not fetched_impact_data and status == "Success":
        print("                      ℹ️  Trying alternative methods as primary failed/yielded no data.")
        if hasattr(calc_result_obj, 'impact_results') and calc_result_obj.impact_results is not None:
            fetched_impact_data = calc_result_obj.impact_results
            data_source_description = "'impact_results' attribute"
            print(f"                      📊 Using '{data_source_description}'. Found {'data' if len(fetched_impact_data) > 0 else 'an empty list'}.")
        elif hasattr(calc_result_obj, 'get_impact_results'):
            try:
                print("                      🔍 Attempting to use 'get_impact_results()'.")
                fetched_impact_data = calc_result_obj.get_impact_results()
                data_source_description = "'get_impact_results()'"
                if fetched_impact_data is not None:
                    print(f"                      📊 '{data_source_description}' call successful, returned {'data' if len(fetched_impact_data) > 0 else 'an empty list'}.")
                else:
                    print(f"                      📊 '{data_source_description}' returned None.")
            except Exception as e:
                print(f"                      ⚠️ Error calling 'get_impact_results()': {e}")
                fetched_impact_data = None
                status = f"Error calling {data_source_description}"
        else:
            print("                      ❌ No alternative methods ('impact_results', 'get_impact_results') yielded data.")
            if status == "Success": status = "No alternative result methods"

    if fetched_impact_data:
        print(f"                      🔬 Found {len(fetched_impact_data)} impact items using {data_source_description}.")
        found_any_category = False
        for item_idx, impact_result_item in enumerate(fetched_impact_data):
            try:
                if impact_result_item is None: continue
                if not hasattr(impact_result_item, 'impact_category'): continue
                if not hasattr(impact_result_item, 'amount'): continue
                
                impact_cat_ref = impact_result_item.impact_category
                item_numerical_value = impact_result_item.amount
                
                if not impact_cat_ref or not hasattr(impact_cat_ref, 'id') or not impact_cat_ref.id: continue
                
                impact_cat_obj_from_res = client.get(olca_schema.ImpactCategory, impact_cat_ref.id)
                if not impact_cat_obj_from_res or not hasattr(impact_cat_obj_from_res, 'name'): continue
                
                actual_impact_cat_name = impact_cat_obj_from_res.name
                ref_unit_name = getattr(impact_cat_obj_from_res, 'reference_unit_name', 'N/A')
                
                all_impact_values[actual_impact_cat_name] = item_numerical_value
                found_any_category = True
                # Less verbose print, only show if specifically debugging, or a summary after loop
                # print(f"                      เก็บข้อมูล: {actual_impact_cat_name} = {item_numerical_value} {ref_unit_name}")

            except Exception as e_item:
                print(f"                      ⚠️ Error processing impact item {item_idx}: {e_item}")
                continue 
        
        if found_any_category:
            print(f"                      📊 Successfully extracted {len(all_impact_values)} impact category values.")
            # If all_impact_values is populated, status should generally be Success unless overridden by an earlier error
            if status not in ["Calc Server Error", "Calc Error After Wait"]: # Don't overwrite critical calc errors
                 status = "Success" # Ensure status is Success if we got some data
        elif status == "Success": # No categories found, but no prior major error
            status = "Impact Categories Not Found in Results"
            print(f"                      ℹ️ No impact categories found among items from {data_source_description}.")

    elif status == "Success": # No fetched_impact_data and no prior major error
        status = "No Impact Data List"
        print(f"                      ❌ No impact data list found from {data_source_description}.")
    
    # If all_impact_values is empty but status is still "Success", it means no data was extractable.
    # Set a more specific status if all_impact_values is empty and status is still "Success".
    if not all_impact_values and status == "Success":
        status = "No Valid Impact Values Extracted"

    return all_impact_values, status


def load_process_names_from_file(filepath):
    process_names = []
    if not os.path.exists(filepath):
        print(f"❌ ERROR: Process names file not found at '{filepath}'")
        return process_names
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                name = line.strip()
                if name:
                    process_names.append(name)
        print(f"📄 Read {len(process_names)} process names from '{filepath}'.")
    except Exception as e:
        print(f"❌ ERROR: Could not read file '{filepath}': {e}")
    return process_names

def analyze_and_create_valid_systems():
    results_data = []
    client = None
    desired_process_names_count = 0
    matched_process_names_count = 0

    # Define default empty values for new JSON columns
    default_impact_values_json = json.dumps({})
    default_input_flows_json = json.dumps([])
    default_output_flows_json = json.dumps([])

    try:
        desired_process_names = load_process_names_from_file(PROCESS_NAMES_FILEPATH)
        desired_process_names_count = len(desired_process_names)
        if not desired_process_names:
            print("❌ No process names loaded. Cannot proceed.")
            results_data.append({
                "Process Name": "File Error", "Product Flow": "N/A", "Unit": "N/A",
                "Carbon Footprint (kg CO2 eq)": None,
                "ImpactValuesJSON": default_impact_values_json,
                "InputFlowsJSON": default_input_flows_json,
                "OutputFlowsJSON": default_output_flows_json,
                "Status": "No process names in file"
            })
            return pd.DataFrame(results_data), desired_process_names_count, matched_process_names_count

        client = olca.Client(OLCA_PORT)
        print("✅ Connected to OpenLCA")
        print("ℹ️ Fetching all process descriptors...")
        all_process_descriptors = client.get_descriptors(olca_schema.Process)
        if not all_process_descriptors:
            print("❌ ERROR: Could not fetch process descriptors.")
            results_data.append({
                "Process Name": "OpenLCA Error", "Product Flow": "N/A", "Unit": "N/A",
                "Carbon Footprint (kg CO2 eq)": None,
                "ImpactValuesJSON": default_impact_values_json,
                "InputFlowsJSON": default_input_flows_json,
                "OutputFlowsJSON": default_output_flows_json,
                "Status": "Failed to fetch process descriptors"
            })
            return pd.DataFrame(results_data), desired_process_names_count, matched_process_names_count

        process_name_to_id_map = {desc.name: desc.id for desc in all_process_descriptors}
        dynamic_target_processes = []
        missing_names_in_db_count = 0
        for name_from_file in desired_process_names:
            if name_from_file in process_name_to_id_map:
                dynamic_target_processes.append({"name": name_from_file, "id": process_name_to_id_map[name_from_file]})
            else:
                missing_names_in_db_count += 1
                results_data.append({
                    "Process Name": name_from_file, "Product Flow": "N/A", "Unit": "N/A",
                    "Carbon Footprint (kg CO2 eq)": None,
                    "ImpactValuesJSON": default_impact_values_json,
                    "InputFlowsJSON": default_input_flows_json,
                    "OutputFlowsJSON": default_output_flows_json,
                    "Status": "Name Not Found in DB"
                })

        matched_process_names_count = len(dynamic_target_processes)
        print(f"✅ Matched {matched_process_names_count}/{desired_process_names_count} names from file.")
        if missing_names_in_db_count > 0: print(f"ℹ️ {missing_names_in_db_count} names from file were not found in DB.")

        if not dynamic_target_processes:
            print("❌ ERROR: No matched processes to analyze.")
            if not results_data: # Should not happen if missing_names_in_db_count handled correctly
                results_data.append({
                    "Process Name": "Matching Error", "Product Flow": "N/A", "Unit": "N/A",
                    "Carbon Footprint (kg CO2 eq)": None,
                    "ImpactValuesJSON": default_impact_values_json,
                    "InputFlowsJSON": default_input_flows_json,
                    "OutputFlowsJSON": default_output_flows_json,
                    "Status": "No names matched"
                })
            return pd.DataFrame(results_data), desired_process_names_count, matched_process_names_count

        lcia_method_full_obj = next((m for m in client.get_all(olca_schema.ImpactMethod) if m.name == LCIA_METHOD_NAME), None)
        if not lcia_method_full_obj:
            print(f"❌ LCIA method '{LCIA_METHOD_NAME}' not found")
            # Add error rows for all matched processes if LCIA method is missing
            for target_info in dynamic_target_processes:
                 results_data.append({
                    "Process Name": target_info['name'], "Product Flow": "N/A", "Unit": "N/A",
                    "Carbon Footprint (kg CO2 eq)": None,
                    "ImpactValuesJSON": default_impact_values_json,
                    "InputFlowsJSON": default_input_flows_json,
                    "OutputFlowsJSON": default_output_flows_json,
                    "Status": "LCIA Method Not Found"
                })
            # Filter out any previous non-process specific errors if any
            results_data = [r for r in results_data if "Process Name" in r and r["Process Name"] != "File Error" and r["Process Name"] != "OpenLCA Error" and r["Process Name"] != "Matching Error"]
            if not results_data : # If all were errors of other types
                 results_data.append({"Process Name": "LCIA Method Error", "Product Flow": "N/A", "Unit": "N/A", "Carbon Footprint (kg CO2 eq)": None, "ImpactValuesJSON": default_impact_values_json, "InputFlowsJSON": default_input_flows_json, "OutputFlowsJSON": default_output_flows_json, "Status": "LCIA Method Not Found (Global)"})

            return pd.DataFrame(results_data), desired_process_names_count, matched_process_names_count
        print(f"✅ Found LCIA method: {lcia_method_full_obj.name} (ID: {lcia_method_full_obj.id})")
        impact_method_ref = olca_schema.Ref(id=lcia_method_full_obj.id, name=lcia_method_full_obj.name, category=getattr(lcia_method_full_obj, 'category', None), ref_type=getattr(olca_schema.ModelType, 'IMPACT_METHOD', getattr(olca_schema.RefType, 'ImpactMethod', None)))

        for i, target_info in enumerate(dynamic_target_processes):
            target_process_id = target_info['id']
            target_process_name_from_list = target_info['name']
            print(f"\n{'='*60}\nProcessing {i+1}/{len(dynamic_target_processes)}: '{target_process_name_from_list}' (ID: {target_process_id})\n{'='*60}")
            
            current_status = "Starting"
            actual_process_name_from_db = "N/A"
            product_flow_name = "N/A" # For the QR
            unit_name_for_df = "N/A" # For the QR
            carbon_footprint_val = None # For the TARGET_IMPACT_CATEGORY
            
            all_impact_results_dict = {}
            impact_values_json_for_csv = default_impact_values_json
            input_flows_list = []
            output_flows_list = []
            input_flows_json = default_input_flows_json
            output_flows_json = default_output_flows_json

            try:
                process = client.get(olca_schema.Process, target_process_id)
                if not process:
                    current_status = "Process Not Found by ID (Post-match)"
                else:
                    actual_process_name_from_db = process.name
                    print(f"    ✅ Retrieved DB process: {actual_process_name_from_db}")
                    if actual_process_name_from_db != target_process_name_from_list: print(f"    ⚠️ Name mismatch: File='{target_process_name_from_list}', DB='{actual_process_name_from_db}'")
                    current_status = "Process Retrieved"

                    # Extract Input and Output Flows
                    if hasattr(process, 'exchanges') and process.exchanges:
                        print(f"    🔍 Analyzing {len(process.exchanges)} exchanges for inputs/outputs...")
                        for ex_idx, ex in enumerate(process.exchanges):
                            flow_name = "N/A"
                            unit_name = "N/A"
                            amount = 0.0
                            flow_type = "Unknown"
                            
                            try:
                                if ex.flow and hasattr(ex.flow, 'name'):
                                    flow_name = ex.flow.name
                                if ex.unit and hasattr(ex.unit, 'name'):
                                    unit_name = ex.unit.name
                                if hasattr(ex, 'amount'):
                                    amount = ex.amount
                                
                                flow_data = {"flow_name": flow_name, "amount": amount, "unit": unit_name}
                                
                                if hasattr(ex, 'is_input') and ex.is_input:
                                    input_flows_list.append(flow_data)
                                    flow_type = "Input"
                                elif hasattr(ex, 'is_input') and not ex.is_input: # Output
                                    output_flows_list.append(flow_data)
                                    flow_type = "Output"
                                # print(f"        Processed Exchange {ex_idx+1}: {flow_type} - {flow_name}, {amount} {unit_name}")

                            except Exception as e_ex:
                                print(f"        ⚠️ Error processing exchange item {ex_idx}: {e_ex}")
                                if hasattr(ex, 'is_input') and ex.is_input:
                                    input_flows_list.append({"flow_name": "Error in exchange", "amount": 0, "unit": str(e_ex)[:50]})
                                elif hasattr(ex, 'is_input') and not ex.is_input:
                                     output_flows_list.append({"flow_name": "Error in exchange", "amount": 0, "unit": str(e_ex)[:50]})


                        input_flows_json = json.dumps(input_flows_list)
                        output_flows_json = json.dumps(output_flows_list)
                        print(f"    📊 Found {len(input_flows_list)} inputs and {len(output_flows_list)} outputs.")
                    else:
                        print("    ℹ️ No exchanges found for this process.")
                        current_status = "Process Has No Exchanges"


                    # Proceed with Product System Creation and Calculation
                    qr_exchange = next((ex for ex in process.exchanges if getattr(ex, 'is_quantitative_reference', False) and not getattr(ex, 'is_input', True)), None)
                    if not qr_exchange:
                        qr_exchange = next((ex for ex in process.exchanges if hasattr(ex, 'is_input') and not ex.is_input), None) # Fallback: first output
                        if qr_exchange: print(f"            ⚠️ Using first output as QR (Fallback)")
                    
                    if not qr_exchange:
                        current_status = "No Reference Output Exchange" if current_status == "Process Retrieved" or current_status == "Process Has No Exchanges" else current_status + "; No QR"
                    else:
                        print(f"            ✅ Found QR: Flow='{qr_exchange.flow.name if qr_exchange.flow else 'N/A'}'")
                        current_status = "QR Found"
                        if not qr_exchange.flow or not qr_exchange.flow.id:
                            current_status = "QR Flow Invalid"
                        else:
                            flow_of_qr = client.get(olca_schema.Flow, qr_exchange.flow.id)
                            product_flow_name = flow_of_qr.name if flow_of_qr and hasattr(flow_of_qr, 'name') else "Flow Error"
                            if not flow_of_qr:
                                current_status = "Flow Error for QR"
                            else:
                                current_status = "Flow Retrieved for QR"
                                ref_flow_property_factor = next((fpf for fpf in flow_of_qr.flow_properties if getattr(fpf, 'is_ref_flow_property', False)), None) if hasattr(flow_of_qr, 'flow_properties') and flow_of_qr.flow_properties else None
                                if not ref_flow_property_factor and hasattr(flow_of_qr, 'flow_properties') and flow_of_qr.flow_properties:
                                    ref_flow_property_factor = flow_of_qr.flow_properties[0]
                                    print(f"                    ⚠️ Using first FlowPropertyFactor as ref.")
                                
                                if not ref_flow_property_factor:
                                    current_status = "No Ref FlowPropertyFactor for QR"
                                elif not ref_flow_property_factor.flow_property or not ref_flow_property_factor.flow_property.id:
                                    current_status = "Ref FlowPropertyFactor Invalid for QR"
                                else:
                                    flow_property_ref_for_ps = ref_flow_property_factor.flow_property
                                    print(f"            ✅ FlowProperty for QR: '{flow_property_ref_for_ps.name}'")
                                    current_status = "FlowProp Found for QR"
                                
                                unit_ref_for_ps = getattr(qr_exchange, 'unit', None)
                                if unit_ref_for_ps and unit_ref_for_ps.id:
                                    unit_name_for_df = unit_ref_for_ps.name if hasattr(unit_ref_for_ps, 'name') else "Unit Name?"
                                    print(f"            ✅ Unit for QR: {unit_name_for_df}")
                                    if current_status == "FlowProp Found for QR": current_status = "Unit&FlowProp Ready for PS"
                                else:
                                    if current_status == "FlowProp Found for QR": current_status = "No Unit on QR for PS"
                        
                        print(f"            📦 Product Flow (QR): {product_flow_name}, 📏 Unit (QR): {unit_name_for_df}")

                        if current_status == "Unit&FlowProp Ready for PS":
                            print(f"            🔧 Creating product system...")
                            ps = olca_schema.ProductSystem()
                            ps_name_prefix = actual_process_name_from_db[:20].replace(' ', '_').replace('/', '_').replace('*','_')
                            ps.name = f"PS_{ps_name_prefix}_{str(uuid.uuid4())[:4]}"
                            process_ref_type = getattr(olca_schema.ModelType, 'PROCESS', getattr(olca_schema.RefType, 'Process', None))
                            ps.processes = [olca_schema.Ref(id=process.id,name=process.name,category=getattr(process, 'category', None),ref_type=process_ref_type)]
                            ps.ref_process = olca_schema.Ref(id=process.id,name=process.name,category=getattr(process, 'category', None),ref_type=process_ref_type)
                            ps.ref_exchange = olca_schema.ExchangeRef(internal_id=qr_exchange.internal_id)
                            ps.target_amount = 1.0
                            ps.target_unit = unit_ref_for_ps
                            ps.target_flow_property = flow_property_ref_for_ps
                            
                            persisted_ps_ref = client.put(ps)
                            if not persisted_ps_ref or not persisted_ps_ref.id:
                                all_ps_desc = client.get_descriptors(olca_schema.ProductSystem)
                                persisted_ps_ref = next((d for d in all_ps_desc if d.name == ps.name), None) if all_ps_desc else None
                                if not persisted_ps_ref : raise Exception(f"Failed to create/retrieve PS '{ps.name}'")
                            print(f"            ✅ PS created: {persisted_ps_ref.name}")
                            print(f"            🧮 Setting up calculation...")

                            setup = olca_schema.CalculationSetup()
                            setup.calculation_type = "SIMPLE_CALCULATION"
                            setup.target = persisted_ps_ref
                            setup.impact_method = impact_method_ref
                            setup.amount = 1.0
                            setup.unit = unit_ref_for_ps # From QR
                            setup.flow_property = flow_property_ref_for_ps # From QR
                            setup.with_impacts = True
                            # setup.with_regionalization = False # Optional, default is false
                            # setup.allocation_method = None # Optional

                            print(f"            ▶️ Running calculation...")
                            calc_result = client.calculate(setup)
                            print(f"            ✅ Calc attempt finished.")
                            
                            all_impact_results_dict, status_from_result = extract_result_safely(client, calc_result)
                            current_status = status_from_result # Update status based on result extraction

                            if all_impact_results_dict: # If we got any impact results
                                impact_values_json_for_csv = json.dumps(all_impact_results_dict)
                                carbon_footprint_val = all_impact_results_dict.get(TARGET_IMPACT_CATEGORY) # Get specific one for old column
                                
                                if carbon_footprint_val is not None:
                                     print(f"            🎉 CF for '{TARGET_IMPACT_CATEGORY}': {carbon_footprint_val}")
                                     print(f"            ℹ️ All impact values ({len(all_impact_results_dict)}) stored in JSON.")
                                     # If we found the target, and other results, status should be success
                                     if current_status not in ["Calc Server Error", "Calc Error After Wait", "Calculation Result Invalid/Empty", "Calculation Result Invalid UID"]:
                                         current_status = "Success"

                                elif current_status == "Success": # all_impact_results_dict is not empty, but TARGET_IMPACT_CATEGORY is not in it
                                    current_status = "Target CF Not In Results"
                                    print(f"            ⚠️ Target CF '{TARGET_IMPACT_CATEGORY}' not found in results, but other impacts present.")

                            # If current_status is still indicating a pre-req missing for PS, but we attempted calculation.
                            # This means calculation failed or yielded no data, so status from extract_result_safely is more relevant.
                            # The status_from_result should have already updated current_status.

                        else: # current_status was not "Unit&FlowProp Ready for PS"
                            print(f"            ❌ Skipping PS creation and calculation due to: {current_status}")
                            # Ensure status reflects the pre-calculation failure if it wasn't updated by calc attempt
                            if not current_status.startswith("Calc") and not current_status.startswith("No Impact"):
                                pass # current_status should already reflect the issue
            
            except Exception as process_error:
                print(f"    ❌ Error in main loop for '{target_process_name_from_list}': {process_error}")
                traceback.print_exc(limit=1)
                current_status = f"Process Error: {str(process_error)[:100]}"
            
            results_data.append({
                "Process Name": target_process_name_from_list,
                "Product Flow": product_flow_name,
                "Unit": unit_name_for_df,
                "Carbon Footprint (kg CO2 eq)": carbon_footprint_val,
                "ImpactValuesJSON": impact_values_json_for_csv,
                "InputFlowsJSON": input_flows_json,
                "OutputFlowsJSON": output_flows_json,
                "Status": current_status
            })

    except ConnectionRefusedError:
        print(f"❌ Connection refused. Ensure OpenLCA IPC server is running on port {OLCA_PORT}.")
        if not results_data:
            results_data.append({
                "Process Name": "Connection Error", "Product Flow": "N/A", "Unit": "N/A",
                "Carbon Footprint (kg CO2 eq)": None,
                "ImpactValuesJSON": default_impact_values_json,
                "InputFlowsJSON": default_input_flows_json,
                "OutputFlowsJSON": default_output_flows_json,
                "Status": "Connection Error"
            })
    except Exception as main_error:
        print(f"❌ Main script error: {main_error}")
        traceback.print_exc()
        if not results_data:
            results_data.append({
                "Process Name": "Script Error", "Product Flow": "N/A", "Unit": "N/A",
                "Carbon Footprint (kg CO2 eq)": None,
                "ImpactValuesJSON": default_impact_values_json,
                "InputFlowsJSON": default_input_flows_json,
                "OutputFlowsJSON": default_output_flows_json,
                "Status": f"Main Error: {str(main_error)[:100]}"
            })
    finally:
        if client: print("\nScript finished processing.")
    
    # Ensure all records have the new columns, even if they were added before the main loop
    final_df = pd.DataFrame(results_data)
    for col, default_val_json in [
        ("ImpactValuesJSON", default_impact_values_json),
        ("InputFlowsJSON", default_input_flows_json),
        ("OutputFlowsJSON", default_output_flows_json)
    ]:
        if col not in final_df.columns:
            final_df[col] = default_val_json
        else:
            final_df[col].fillna(default_val_json, inplace=True)
            # Ensure existing non-null values that might not be JSON (e.g. if error occurred before JSON creation) are also handled
            # This is a bit broad, ideally handled by ensuring default JSONs are used when row is first created.
            # For now, this ensures valid JSON strings for outputs.
            def make_sure_is_json_string(val, default_json):
                if pd.isna(val): return default_json
                try:
                    json.loads(val) # check if it's already a valid json string
                    return val
                except (TypeError, json.JSONDecodeError): # if not a string or not valid json
                    if isinstance(val, (dict, list)): # if it's a dict/list, dump it
                        return json.dumps(val)
                    return default_json # fallback for other types or unparsable strings
            
            # final_df[col] = final_df[col].apply(lambda x: make_sure_is_json_string(x, default_val_json))


    return final_df, desired_process_names_count, matched_process_names_count

if __name__ == "__main__":
    print("🚀 OpenLCA Data Extraction - Enhanced Script")
    print("=" * 60)
    
    # Ensure PROCESS_NAMES_FILEPATH exists
    if not os.path.exists(PROCESS_NAMES_FILEPATH):
        print(f"❌ CRITICAL ERROR: The process names file '{PROCESS_NAMES_FILEPATH}' was not found.")
        print("Please create this file with one process name per line, or check the path.")
        # Create a dummy file if it doesn't exist to prevent immediate script crash in load_process_names_from_file
        # but the script will then report 0 processes from file.
        # open(PROCESS_NAMES_FILEPATH, 'a').close() # This might hide the error too much. Better to let it fail if critical.
    
    results_df, total_from_file, total_matched = analyze_and_create_valid_systems()
    
    if not results_df.empty:
        print(f"\n📋 FINAL RESULTS")
        print("=" * 60)
        with pd.option_context('display.max_rows', 10, 'display.max_columns', None, 'display.width', 1000): # Show only a few rows in console
            print(results_df)
        
        filename = "openlca_enhanced_results.csv" # New filename
        try:
            results_df.to_csv(filename, index=False)
            print(f"\n💾 Saved to: {filename}")
        except Exception as e_csv:
            print(f"\n⚠️ Error saving CSV: {e_csv}")
            
        print(f"\n📊 Summary:")
        print(f"  - Names read from file: {total_from_file}")
        print(f"  - Names matched to DB: {total_matched}")
        
        # Recalculate successful_calculations based on the updated status meaning
        # "Success" means full calculation and primary CF target found.
        # "Target CF Not In Results" also means calc was successful but primary target missing.
        # "No Valid Impact Values Extracted" means calc ran but no impact values at all.
        successful_calculations = len(results_df[results_df['Status'].isin(['Success', 'Target CF Not In Results'])])
        attempted_calculations = len(results_df[results_df['Status'].apply(lambda s: isinstance(s, str) and (s.startswith("Calc") or s.startswith("Success") or s.startswith("Target CF") or s.startswith("No Impact") or s.startswith("No Valid") or s == "Unit&FlowProp Ready for PS"))])


        print(f"  - Successful calculations (at least some impact data): {successful_calculations} / {attempted_calculations if attempted_calculations > 0 else (total_matched if total_matched > 0 else 'N/A attempted')}")
        
        status_counts = results_df['Status'].value_counts()
        if not status_counts.empty:
            print("  - Breakdown of all statuses:")
            for status_name, count_val in status_counts.items():
                print(f"    - {status_name}: {count_val}")
    else:
        print("❌ No results generated (or initial file/DB error). Check logs above.")

🚀 OpenLCA Data Extraction - Enhanced Script
📄 Read 957 process names from 'all_process_names.txt'.
✅ Connected to OpenLCA
ℹ️ Fetching all process descriptors...
✅ Matched 891/957 names from file.
ℹ️ 66 names from file were not found in DB.
✅ Found LCIA method: IPCC 2021 (ISO 14067) (ID: 437c5849-901b-4eba-af18-211f849bde6a)

Processing 1/891: '6000-Q GasCogenerate Power Used in Place/MJ S' (ID: ea9c9c23-7db9-3a2a-b3f1-7a68d3fc7c6e)
    ✅ Retrieved DB process: 6000-Q GasCogenerate Power Used in Place/MJ S
    🔍 Analyzing 20 exchanges for inputs/outputs...
    📊 Found 2 inputs and 18 outputs.
            ✅ Found QR: Flow='6000-Q GasCogenerate Power Used in Place/MJ S'
            ✅ FlowProperty for QR: 'Energy'
            ✅ Unit for QR: MJ
            📦 Product Flow (QR): 6000-Q GasCogenerate Power Used in Place/MJ S, 📏 Unit (QR): MJ
            🔧 Creating product system...
            ✅ PS created: PS_6000-Q_GasCogenerate_9d5f
            🧮 Setting up calculation...
            ▶️ Runn

ERROR:root:request total-impacts failed: 500: Failed to call method public org.openlca.ipc.RpcResponse org.openlca.ipc.handlers.ResultHandler.getTotalImpacts(org.openlca.ipc.RpcRequest): null


                      🔍 Attempting to use 'get_total_impacts()'.
                      📊 ''get_total_impacts()'' call successful, returned an empty list.
                      ℹ️  Trying alternative methods as primary failed/yielded no data.
                      ❌ No alternative methods ('impact_results', 'get_impact_results') yielded data.

Processing 3/891: '6142-Q BioFuel Power Used in Place/MJ S' (ID: d8c9eedf-a3fc-31a6-b1b8-29ab1d76ea18)
    ✅ Retrieved DB process: 6142-Q BioFuel Power Used in Place/MJ S
    🔍 Analyzing 94 exchanges for inputs/outputs...
    📊 Found 22 inputs and 72 outputs.
            ✅ Found QR: Flow='6142-Q BioFuel Power Used in Place/MJ S'
            ✅ FlowProperty for QR: 'Energy'
            ✅ Unit for QR: MJ
            📦 Product Flow (QR): 6142-Q BioFuel Power Used in Place/MJ S, 📏 Unit (QR): MJ
            🔧 Creating product system...
            ✅ PS created: PS_6142-Q_BioFuel_Power_8003
            🧮 Setting up calculation...
            ▶️ Running c

ERROR:root:request total-impacts failed: 500: Failed to call method public org.openlca.ipc.RpcResponse org.openlca.ipc.handlers.ResultHandler.getTotalImpacts(org.openlca.ipc.RpcRequest): null


                      🔍 Attempting to use 'get_total_impacts()'.
                      📊 ''get_total_impacts()'' call successful, returned an empty list.
                      ℹ️  Trying alternative methods as primary failed/yielded no data.
                      ❌ No alternative methods ('impact_results', 'get_impact_results') yielded data.

Processing 7/891: '6266-Q Wind Power Used  in Place*MJ S' (ID: 70e706c6-e9ad-34c7-a63e-aa6b792a5bff)
    ✅ Retrieved DB process: 6266-Q Wind Power Used  in Place*MJ S
    🔍 Analyzing 1 exchanges for inputs/outputs...
    📊 Found 0 inputs and 1 outputs.
            ✅ Found QR: Flow='6266-Q Wind Power Used  in Place*MJ S'
            ✅ FlowProperty for QR: 'Energy'
            ✅ Unit for QR: MJ
            📦 Product Flow (QR): 6266-Q Wind Power Used  in Place*MJ S, 📏 Unit (QR): MJ
            🔧 Creating product system...
            ✅ PS created: PS_6266-Q_Wind_Power_Us_3e1c
            🧮 Setting up calculation...
            ▶️ Running calculation.

ERROR:root:request total-impacts failed: 500: Failed to call method public org.openlca.ipc.RpcResponse org.openlca.ipc.handlers.ResultHandler.getTotalImpacts(org.openlca.ipc.RpcRequest): null


                      🔍 Attempting to use 'get_total_impacts()'.
                      📊 ''get_total_impacts()'' call successful, returned an empty list.
                      ℹ️  Trying alternative methods as primary failed/yielded no data.
                      ❌ No alternative methods ('impact_results', 'get_impact_results') yielded data.

Processing 8/891: '6331-AU Generate Steam on Site/MJ S' (ID: 9d8a4c10-cbd5-3416-84df-f169f0890dca)
    ✅ Retrieved DB process: 6331-AU Generate Steam on Site/MJ S
    🔍 Analyzing 197 exchanges for inputs/outputs...
    📊 Found 77 inputs and 120 outputs.
            ✅ Found QR: Flow='6331-AU Generate Steam on Site/MJ S'
            ✅ FlowProperty for QR: 'Energy'
            ✅ Unit for QR: MJ
            📦 Product Flow (QR): 6331-AU Generate Steam on Site/MJ S, 📏 Unit (QR): MJ
            🔧 Creating product system...
            ✅ PS created: PS_6331-AU_Generate_Ste_326c
            🧮 Setting up calculation...
            ▶️ Running calculation...


C:\Users\artur\AppData\Local\Temp\ipykernel_30620\1774800607.py:483: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_df[col].fillna(default_val_json, inplace=True)



💾 Saved to: openlca_enhanced_results.csv

📊 Summary:
  - Names read from file: 957
  - Names matched to DB: 891
  - Successful calculations (at least some impact data): 888 / 888
  - Breakdown of all statuses:
    - Success: 888
    - Name Not Found in DB: 66
    - No alternative result methods: 3
